# 08. Compare Models

Compares GMF, NeuMF, and Popularity on HR@10 and NDCG@10 metrics and exports production candidate artifacts with SHA256 checksums.

In [ ]:
import os
import numpy as np
import pandas as pd
import json
import shutil
import hashlib

outputs_dir = os.path.join("..", "..", "outputs", "recommendation")
artifacts_dir = os.path.join("..", "..", "artifacts", "recommendation")
saved_models_dir = os.path.join("..", "..", "saved_models", "recommendation")
processed_dir = os.path.join("..", "..", "datasets", "processed")

with open(os.path.join(outputs_dir, "pop_metrics.json"), "r") as f:
    pop = json.load(f)
with open(os.path.join(outputs_dir, "gmf_metrics.json"), "r") as f:
    gmf = json.load(f)
with open(os.path.join(outputs_dir, "neumf_metrics.json"), "r") as f:
    neumf = json.load(f)

metrics_df = pd.DataFrame({
    "Algorithm": ["Popularity", "GMF", "NeuMF"],
    "HR@10": [pop["popularity_hr_10"], gmf["gmf_hr_10"], neumf["neumf_hr_10"]],
    "NDCG@10": [pop["popularity_ndcg_10"], gmf["gmf_ndcg_10"], neumf["neumf_ndcg_10"]]
})

print("Model Comparison Table:")
print(metrics_df)

metrics_df.to_csv(os.path.join(artifacts_dir, "recommendation_model_comparison.csv"), index=False)

# Determine the production candidate based on HR@10 (using NDCG@10 as a tie breaker)
candidates_hr = {
    "Popularity": pop["popularity_hr_10"],
    "GMF": gmf["gmf_hr_10"],
    "NeuMF": neumf["neumf_hr_10"]
}
candidates_ndcg = {
    "Popularity": pop["popularity_ndcg_10"],
    "GMF": gmf["gmf_ndcg_10"],
    "NeuMF": neumf["neumf_ndcg_10"]
}

# Sort candidates first by HR@10 descending, then NDCG@10 descending
winner = sorted(candidates_hr.keys(), key=lambda k: (candidates_hr[k], candidates_ndcg[k]), reverse=True)[0]
print(f"\nSelected production candidate: {winner}")

train_df = pd.read_csv(os.path.join(processed_dir, "train.csv"))
pop_scores = train_df["item_idx"].value_counts().to_dict()

# Copy selected model to model.keras or write popularity json
if winner == "NeuMF":
    shutil.copy2(os.path.join(artifacts_dir, "neumf_model.keras"), os.path.join(artifacts_dir, "model.keras"))
    shutil.copy2(os.path.join(artifacts_dir, "neumf_model.keras"), os.path.join(saved_models_dir, "model.keras"))
elif winner == "GMF":
    shutil.copy2(os.path.join(artifacts_dir, "gmf_model.keras"), os.path.join(artifacts_dir, "model.keras"))
    shutil.copy2(os.path.join(artifacts_dir, "gmf_model.keras"), os.path.join(saved_models_dir, "model.keras"))
else:
    # Fallback/Popularity (just save popularity dict as model config or similar)
    with open(os.path.join(artifacts_dir, "model.json"), "w") as f:
        json.dump({"type": "Popularity", "scores": {int(k): int(v) for k, v in pop_scores.items()}}, f)
    with open(os.path.join(saved_models_dir, "model.json"), "w") as f:
        json.dump({"type": "Popularity", "scores": {int(k): int(v) for k, v in pop_scores.items()}}, f)

full_df = pd.read_csv(os.path.join(processed_dir, "recommendation_processed.csv"))
num_users = int(full_df["user_idx"].max() + 1)
num_items = int(full_df["item_idx"].max() + 1)

# Create metadata.json
metadata = {
    "model_type": winner,
    "num_users": num_users,
    "num_items": num_items,
    "metrics": {
        "HR@10": candidates_hr[winner],
        "NDCG@10": candidates_ndcg[winner]
    }
}
with open(os.path.join(artifacts_dir, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=4)
with open(os.path.join(saved_models_dir, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=4)

# Create training_config.json
training_config = {
    "epochs": 3,
    "batch_size": 256,
    "optimizer": "adam",
    "loss": "binary_crossentropy",
    "latent_dim_gmf": 16,
    "latent_dim_mlp": 32,
    "negative_ratio": 4
}
with open(os.path.join(artifacts_dir, "training_config.json"), "w") as f:
    json.dump(training_config, f, indent=4)
with open(os.path.join(saved_models_dir, "training_config.json"), "w") as f:
    json.dump(training_config, f, indent=4)

# Create recommendation_metrics.json
rec_metrics = {
    "Popularity": {"HR@10": pop["popularity_hr_10"], "NDCG@10": pop["popularity_ndcg_10"]},
    "GMF": {"HR@10": gmf["gmf_hr_10"], "NDCG@10": gmf["gmf_ndcg_10"], "best_epoch": gmf["gmf_best_epoch"]},
    "NeuMF": {"HR@10": neumf["neumf_hr_10"], "NDCG@10": neumf["neumf_ndcg_10"], "best_epoch": neumf["neumf_best_epoch"]}
}
with open(os.path.join(artifacts_dir, "recommendation_metrics.json"), "w") as f:
    json.dump(rec_metrics, f, indent=4)
with open(os.path.join(saved_models_dir, "recommendation_metrics.json"), "w") as f:
    json.dump(rec_metrics, f, indent=4)

# Calculate SHA256 checksums
def get_sha256_and_size(filepath):
    sha256_hash = hashlib.sha256()
    with open(filepath, "rb") as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest(), os.path.getsize(filepath)

checksums = {}
for f in os.listdir(artifacts_dir):
    if f == "checksums.json":
        continue
    f_path = os.path.join(artifacts_dir, f)
    if os.path.isfile(f_path):
        sha, size = get_sha256_and_size(f_path)
        checksums[f] = {
            "relative_path": f"ml-training/artifacts/recommendation/{f}",
            "sha256": sha,
            "size_bytes": size
        }

with open(os.path.join(artifacts_dir, "checksums.json"), "w") as f:
    json.dump(checksums, f, indent=4)

print("Artifact generation and checksum calculation complete.")